[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aditya-siraskar/PINN_FTRE/blob/main/notebooks/01_phase1_stage2a_reference.ipynb)

# 01 · Stage 2a: ground truth (exact vs FD vs Monte Carlo)

**Phase:** Phase 1: validation (1-D)

**Goal:** Make sure the three reference routes agree before any PINN is trained (Box A, the maths check).

**Outputs:** `figures/stage2_reference/fig01–fig05`, `results/stage2_reference/metrics.json`

> Runtime → Change runtime type → **T4 GPU**. Run the setup cell first. With `USE_DRIVE = True` the repo lives in
> `MyDrive/PINN_FTRE`, so figures and results persist. Core logic lives in `pinn_tailrisk/` and `scripts/`;
> this notebook only runs it and shows the results.

In [ ]:
# @title Setup: repo on Google Drive, install, GPU check  { display-mode: "form" }
USE_DRIVE = True   # keep figures/results across Colab sessions (recommended)
REPO_URL = "https://github.com/aditya-siraskar/PINN_FTRE"

import os, sys
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = "/content/drive/MyDrive"
else:
    BASE = "/content"
ROOT = f"{BASE}/PINN_FTRE"
if not os.path.exists(ROOT):
    !git clone -q {REPO_URL} "{ROOT}"
else:
    rc = os.system(f'git -C "{ROOT}" pull -q --ff-only')
    if rc != 0:   # usually: result files on Drive that have since been committed to GitHub
        print("WARNING: git pull failed; the Drive copy is out of date. To sync (keeps untracked files), run:")
        print(f'!git -C "{ROOT}" fetch -q origin && git -C "{ROOT}" reset -q --hard origin/main')
%cd "{ROOT}"
!pip install -q -e .
sys.path.insert(0, ROOT)

import torch
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (Runtime > Change runtime type > T4 GPU)")

In [ ]:
# autoreload picks up library edits without a restart. Colab's IPython build imports the
# removed `imp` module on Python >= 3.12, so it is optional: if it is skipped, use
# Runtime > Restart session after editing pinn_tailrisk/.
try:
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")
except ModuleNotFoundError:
    print("autoreload unavailable here; restart the session after editing the library")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from pinn_tailrisk import plotting as P
from pinn_tailrisk.notebook import show_figures, load_metrics
P.use_style()

## 1. Configuration
The ground truth shared by every stage (`configs/base.yaml`).

In [ ]:
print(open('configs/base.yaml').read())

## 2. Run the stage
Takes a few minutes (fig05 repeats Monte Carlo 200× per N).

In [ ]:
!python scripts/stage2_reference.py

## 3. Figures and what to check

| Figure | Pass criterion |
|---|---|
| fig01_target_density | the log-scale panel shows the jump-fattened left tail vs the Gaussian |
| fig02_fd_mc_validation | FD and MC lie on the exact curve down to ~1e-5; FD log-error below 1e-3 where p > 1e-6 |
| fig03_fd_convergence | L¹ and ES₉₉ errors fall with slope ≈ 2 (FD runs on the wide `fd_domain`, so no truncation floor) |
| fig04_var_es_vs_alpha | Gaussian ES₉₉ clearly below the jump-diffusion ES₉₉ |
| fig05_mc_cost | MC error ∝ N^(−1/2); MC cost grows linearly. This is the motivation for RQ5 |

In [ ]:
show_figures("stage2_reference")

## 4. Numbers

In [ ]:
m = load_metrics("stage2_reference")
rows = [dict(horizon=h, alpha=a, VaR_exact=v["exact"][0], VaR_fd=v["fd"][0], ES_exact=v["exact"][1], ES_fd=v["fd"][1])
        for h, d in m["fd_reference"]["var_es"].items() for a, v in d.items()]
df = pd.DataFrame(rows); df["ES_rel_err_fd"] = (df.ES_fd - df.ES_exact) / df.ES_exact
display(df.style.format(precision=5))
display(pd.DataFrame(m["mc_convergence"]))